# Build Consultant Dictionary

Builds `consultant_dictionary.csv` — environmental / water-resources **consulting firms**
that appear in California water, SGMA, and CEQA/NEPA contexts (the firms that author
Groundwater Sustainability Plans, water-rights filings, and environmental review documents).

Every row is labeled with a single flat `entity_type` = `consulting_firm`. There is intentionally
**no sub-classification** by discipline (hydrogeology vs. engineering vs. environmental etc.) —
the dictionary just needs to recognize *that a name is a consulting firm*.

Each output row has:
- `all_names`: pipe-separated canonical name + aliases (the R pipeline splits on `|`)
- `entity_type`: always `consulting_firm`
- `notes`: source provenance + the firm's listed services/qualifications where available

## Sources (live fetch)
This dictionary is an **aggregator** over several public consultant rosters. Each source has its
own small parser; add a new roster by writing one `parse_*` function and appending it to `SOURCES`.

| tag | source | ~count | notes |
|-----|--------|--------|-------|
| `swrcb` | CA State Water Resources Control Board — Water Rights consultants list | ~97 | clean HTML table (firm / services / phone / contacts) |
| `slo`   | San Luis Obispo County — qualified environmental consultants | ~123 | clean HTML table (name / address / phone / qualifications) |
| `lacity`| City of Los Angeles Planning — approved environmental consultant list | ~25 | `<li>` block, firms linked via Constant-Contact tracking host |
| `ehsmatch` | ehsmatch.com EHS/environmental consultant marketplace | ~2700 | **optional** (see `INCLUDE_EHSMATCH`); enumerated via sitemap, names from profile `<h1>` |

### Sources known but not yet wired (drop-in candidates)
- City of San Jose approved environmental consultants (blocks scripted fetch with 403; needs a
  browser-style request or manual snapshot).
- Other city/county "approved/qualified environmental consultant" rosters — there are many; each
  is a new `parse_*` + `SOURCES` entry.

## Cross-dict overlap
Like the sibling dictionaries, this one is intentionally **not** deduped against the other
`core_code/dicts` files at build time. Each dict aims to be a self-contained, comprehensive list
of its category. Cross-dict alias collisions are resolved at runtime in step4's `global_dict`
construction. Duplicate firms *within* this dictionary (same firm on multiple rosters) ARE merged
here into one row with all surface forms as aliases.

In [1]:
import os, re, csv, io, html, time, urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

DICTS_DIR = "."
OUT_CSV   = os.path.join(DICTS_DIR, "consultant_dictionary.csv")

# ehsmatch.com is a large general EHS marketplace (mold/lead/IH firms mixed in with
# water/environmental consultants) and its filtered listing sits behind an auth-gated
# server action, so we can only enumerate the FULL directory via its sitemap. It is
# therefore opt-in. With INCLUDE_EHSMATCH=True and EHS_MAX_PROFILES=None this crawls
# every profile in the sitemap (~2,700 polite, rate-limited requests -> ~20-30 min).
# Set EHS_MAX_PROFILES to an int to cap it, or INCLUDE_EHSMATCH=False to skip entirely.
INCLUDE_EHSMATCH  = True
EHS_MAX_PROFILES  = None         # None = fetch every sitemap profile

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36")

def fetch(url, tries=3):
    """GET a URL as decoded text, with a browser UA and simple retry."""
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.read().decode("utf-8", "replace")
        except Exception as e:
            last = e
            time.sleep(1.0 * (i + 1))
    raise last

## Shared parsing helpers

`first_block` pulls the firm name out of a table cell that often glues the mailing address
onto the name (some rows use `Name<br>street<br>city`, others `<p>Name</p>street`). Treating
both `</p>` and `<br>` as line breaks and taking the first non-empty line handles both.

`norm_key` is the dedup key: lower-cased, punctuation-stripped, with trailing legal suffixes
(Inc/LLC/LLP/Corp/…) removed so `AECOM, Inc.` and `AECOM` collapse, while distinct firms that
merely share a first word stay separate (we key on the whole core name, not a prefix).

In [2]:
def strip_tags(s):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", s))).strip()

def first_block(cell):
    """First non-empty line of a cell, with </p> and <br> treated as breaks."""
    s = re.sub(r"</p>|<br\s*/?>", "\n", cell, flags=re.I)
    s = html.unescape(re.sub(r"<[^>]+>", " ", s))
    for line in s.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            return line.rstrip(",").strip()   # drop a trailing name/address comma
    return ""

def parse_html_table(page_html, name_hdr, note_col, min_cols=4):
    """Generic 4+-column roster table -> list of (firm_name, note_text).
    name_hdr is the lower-cased header text of the name column, used to drop the
    header row; note_col is the 0-based column whose text becomes the note."""
    rows = []
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", page_html, re.S | re.I):
        tds = re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S | re.I)
        if len(tds) < min_cols:
            continue
        name = first_block(tds[0])
        if not name or name.lower() == name_hdr:
            continue
        note = strip_tags(tds[note_col]) if note_col < len(tds) else ""
        rows.append((name, note))
    return rows

_LEGAL = r"(?:,?\s+(?:inc|incorporated|llc|l\.l\.c|llp|lp|ltd|pllc|pc|corp|corporation|co|company|a corporation))\.?$"
def norm_key(name):
    k = html.unescape(name).lower()
    k = k.replace("&", " and ")
    k = re.sub(r"[^a-z0-9 ]+", " ", k)
    k = re.sub(r"\s+", " ", k).strip()
    prev = None
    while prev != k:                       # strip stacked suffixes: 'x, inc, llc'
        prev = k
        k = re.sub(_LEGAL, "", k).strip()
    return k

## Source 1 — CA State Water Resources Control Board (Water Rights consultants)

A stable HTML table: FIRM NAME / SERVICES OFFERED / PHONE / CONTACTS. The firm name is the
first block of column 0; the services text becomes the note.

In [3]:
SWRCB_URL = "https://www.waterboards.ca.gov/waterrights/board_info/consultants_list.html"

def parse_swrcb():
    h = fetch(SWRCB_URL)
    out = []
    for name, services in parse_html_table(h, name_hdr="firm name", note_col=1):
        note = f"services: {services}" if services else ""
        out.append((name, note))
    return out

swrcb_rows = parse_swrcb()
print(f"swrcb: {len(swrcb_rows)} firms  (e.g. {swrcb_rows[0][0]!r} ... {swrcb_rows[-1][0]!r})")

swrcb: 97 firms  (e.g. '4 Creeks, Inc.' ... 'Zanjero')


## Source 2 — San Luis Obispo County (qualified environmental consultants)

HTML table: Consultant Name / Address / Telephone / Qualifications. Name is column 0,
qualifications (column 3) become the note.

In [4]:
SLO_URL = ("https://www.slocounty.ca.gov/departments/planning-building/"
           "how-to-apply-for-a-permit-in-unincorporated-slo-co/"
           "land-use,-subdivision,-zoning/support-services/environmental-review/"
           "qualified-environmental-consultants")

def parse_slo():
    h = fetch(SLO_URL)
    out = []
    for name, quals in parse_html_table(h, name_hdr="consultant name", note_col=3):
        note = f"qualifications: {quals}" if quals else ""
        out.append((name, note))
    return out

slo_rows = parse_slo()
print(f"slo: {len(slo_rows)} firms  (e.g. {slo_rows[0][0]!r} ... {slo_rows[-1][0]!r})")

slo: 123 firms  (e.g. '45dB Acoustics, LLC' ... 'WRA Environmental Consultants')


## Source 3 — City of Los Angeles Planning (approved environmental consultants)

No table here — the firms are a run of `<li>` items, each wrapped in a Constant-Contact
tracking link (`r20.rs6.net`) pointing at the firm's website. That host is the discriminator
that separates firm items from the page's navigation `<li>`s, so the parser survives edits to
the list as long as firms keep their linked websites.

In [5]:
LACITY_URL = "https://planning.lacity.gov/development-services/environmental-review/environmental-consultant-list"
LACITY_LINK_HOST = "r20.rs6.net"   # Constant-Contact tracking host used by firm links

def parse_lacity():
    h = fetch(LACITY_URL)
    out = []
    for li in re.findall(r"<li[^>]*>(.*?)</li>", h, re.S | re.I):
        if LACITY_LINK_HOST not in li:
            continue
        name = strip_tags(li)
        if name:
            out.append((name, ""))
    return out

lacity_rows = parse_lacity()
print(f"lacity: {len(lacity_rows)} firms  (e.g. {lacity_rows[0][0]!r} ... {lacity_rows[-1][0]!r})")

lacity: 25 firms  (e.g. 'AECOM Technical Services, Inc.' ... 'Wood Environment & Infrastructure Solutions, Inc.')


## Source 4 — ehsmatch.com marketplace (optional)

`ehsmatch.com` is a large EHS/environmental consultant marketplace (~2,700 firms). Its
specialty-filtered listing is served by an auth-gated internal API and a JS "Load more" server
action, neither of which is publicly scrapeable — but `robots.txt` permits crawling and the
`sitemap.xml` enumerates every `/c/<slug>` company profile. We take the slugs from the sitemap
and read each firm's real display name from its profile `<h1>` / `<title>`.

Because this is (a) the *unfiltered* directory — it includes mold/lead/industrial-hygiene firms
that aren't water consultants — and (b) one HTTP request per profile, it is **opt-in** via
`INCLUDE_EHSMATCH` and rate-limited by `EHS_MAX_PROFILES` / `EHS_FETCH_DELAY_S`.

In [6]:
EHS_SITEMAP = "https://ehsmatch.com/sitemap.xml"

def ehsmatch_slugs():
    xml = fetch(EHS_SITEMAP)
    slugs = re.findall(r"/c/([a-z0-9-]+)", xml)
    seen, out = set(), []
    for s in slugs:
        if s not in seen:
            seen.add(s); out.append(s)
    return out

def ehsmatch_name(slug):
    """Clean firm name from a profile page; fall back to a title-cased slug."""
    try:
        h = fetch(f"https://ehsmatch.com/c/{slug}")
        m = re.search(r"<h1[^>]*>(.*?)</h1>", h, re.S | re.I)
        if m:
            name = strip_tags(m.group(1))
            if name:
                return name
        m = re.search(r"<title>(.*?)</title>", h, re.S | re.I)
        if m:
            return strip_tags(m.group(1)).split("\u2014")[0].strip()
    except Exception as e:
        print(f"  ! {slug}: {e}")
    return slug.replace("-", " ").title()

def parse_ehsmatch(workers=5):
    # ~2,700 profiles, so fetch names with a small thread pool (keeps the crawl
    # polite but finishes in minutes, not ~30). Order is preserved via an index.
    slugs = ehsmatch_slugs()
    if EHS_MAX_PROFILES is not None:
        slugs = slugs[:EHS_MAX_PROFILES]
    print(f"  fetching {len(slugs)} ehsmatch profiles with {workers} workers ...")
    out = [None] * len(slugs)
    done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(ehsmatch_name, s): i for i, s in enumerate(slugs)}
        for fut in as_completed(futs):
            out[futs[fut]] = (fut.result(), "")
            done += 1
            if done % 500 == 0:
                print(f"    {done}/{len(slugs)}")
    return out

if INCLUDE_EHSMATCH:
    ehsmatch_rows = parse_ehsmatch()
    print(f"ehsmatch: {len(ehsmatch_rows)} firms")
else:
    ehsmatch_rows = []
    print("ehsmatch: skipped (INCLUDE_EHSMATCH=False)")

  fetching 2747 ehsmatch profiles with 5 workers ...


    500/2747


    1000/2747


    1500/2747


    2000/2747


    2500/2747


ehsmatch: 2747 firms


## Combine, dedup, and write

All sources feed one list of `(firm_name, note, source_tag)` records. Records are grouped by
`norm_key`; within a group the longest surface form becomes the canonical name and the rest
become `|`-aliases, with notes concatenated (source-tagged). One row per firm.

In [7]:
SOURCES = [
    ("swrcb",    swrcb_rows),
    ("slo",      slo_rows),
    ("lacity",   lacity_rows),
    ("ehsmatch", ehsmatch_rows),
]

groups = {}   # norm_key -> {'surfaces': {name: None}, 'notes': [], 'sources': set()}
for tag, rows in SOURCES:
    for name, note in rows:
        name = re.sub(r"\s+", " ", name).strip()
        if not name:
            continue
        k = norm_key(name)
        if not k:
            continue
        g = groups.setdefault(k, {"surfaces": {}, "notes": [], "sources": set()})
        g["surfaces"].setdefault(name, None)
        g["sources"].add(tag)
        if note:
            tagged = f"[{tag}] {note}"
            if tagged not in g["notes"]:
                g["notes"].append(tagged)

out_rows = []
for k, g in groups.items():
    # canonical = longest surface form; aliases = the rest (longest-first)
    surfaces = sorted(g["surfaces"].keys(), key=lambda s: (-len(s), s))
    canonical = surfaces[0]
    aliases = [s for s in surfaces[1:]]
    all_names = "|".join([canonical] + aliases)
    src = "+".join(sorted(g["sources"]))
    note = "; ".join(g["notes"])
    note = f"source: {src}" + (f"; {note}" if note else "")
    out_rows.append((all_names, "consulting_firm", note))

out_rows.sort(key=lambda r: r[0].lower())

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["all_names", "entity_type", "notes"])
    for row in out_rows:
        w.writerow(row)

n_alias = sum(1 for r in out_rows if "|" in r[0])
print(f"wrote {OUT_CSV} \u2014 {len(out_rows)} unique consulting firms "
      f"({n_alias} with cross-roster aliases)")
print("  by source combo:")
from collections import Counter
combo = Counter(r[2].split(";")[0].replace("source: ", "") for r in out_rows)
for c, n in combo.most_common():
    print(f"    {c:24s} {n}")

wrote ./consultant_dictionary.csv — 2309 unique consulting firms (92 with cross-roster aliases)
  by source combo:
    ehsmatch                 2083
    slo                      103
    swrcb                    73
    lacity                   13
    ehsmatch+slo             10
    ehsmatch+swrcb           9
    ehsmatch+lacity          6
    ehsmatch+slo+swrcb       4
    ehsmatch+lacity+slo+swrcb 2
    slo+swrcb                2
    lacity+slo               1
    ehsmatch+lacity+swrcb    1
    ehsmatch+lacity+slo      1
    lacity+swrcb             1
